In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse
import itertools
import matplotlib
from matplotlib import cm
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, LinearSegmentedColormap
from datetime import date
import re
# from tqdm import tqdm
import time
# from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr, rankdata, describe, mode, fisher_exact
import scipy as sp
import scipy.stats as stats
from scipy.spatial.distance import jaccard, euclidean, cosine, pdist
# import functools
# import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.metrics import pairwise_distances
import sys
import os
from functools import partial
from contextlib import closing
from multiprocessing import Pool
import re
import h5py
import tabix
import upsetplot
from wordcloud import WordCloud
from sklearn.decomposition import NMF, non_negative_factorization
from collections import Counter
from scipy.stats import hypergeom
from scipy.stats.mstats import gmean
from scipy.cluster import hierarchy
import squarify
try:
    import piplite
    await piplite.install(['ipywidgets'])
except ImportError:
    pass
import ipywidgets as widgets
import json
from subprocess import Popen, PIPE
import requests

today = date.today()

**Conversions for genes to genesets**

In [13]:
# geneset paths
datadir = '/home/ctrader/MSigDB/'
c5gobp_f = datadir + 'c5.go.bp.v2023.1.Hs.symbols.gmt'

# gene path
datadir = '/home/ctrader/region_similarity/data/rank_based/'
gene_f = datadir + 'GREATv4/GREATv4.genes.hg38.tsv'

In [14]:
# build dictionary of genesets mapping to a list of genes
geneset2genes = {}
for gs_f in [c5gobp_f]:
    f = open(gs_f)
    for line in f.readlines():
        line = line.strip().split('\t')
        gs = line[0]
        genes = line[2:]
        geneset2genes[gs] = genes
    f.close()

In [15]:
# build dictionary of genes mapping to a list of genesets
gene2genesets = {}
for gs in geneset2genes.keys():
    genes = geneset2genes[gs]
    for g in genes:
        if g not in gene2genesets:
            gene2genesets[g] = np.array([gs])
        else:
            gene2genesets[g] = np.append(gene2genesets[g], np.array([gs]))

**Mapping genes to search space regions**

In [23]:
# for order in [4, 5, 6, 7, 8, 9, 10, 11]:
#     print(order)
#     datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/hilbert_sfc/'
#     search_space_file = datadir + 'hilbert_regions_order_{}_CT20230718.dhs.cs.sfc.ranks.search_space.ft_2.rt_21.npz'.format(order)
#     search_space_coords = np.load(search_space_file)['coordinates']
#     search_space_ori_inds = np.load(search_space_file)['original_inds']
    
#     search_space_out = np.hstack((search_space_coords, search_space_ori_inds.reshape(-1, 1)))
#     datadir = '/home/ctrader/region_similarity/data/rank_based/hilbert/sfc/20230718/'
#     search_space_out_file = datadir + 'hilbert_regions_order_{}_CT20230718.dhs.cs.sfc.ranks.search_space.ft_2.rt_21.bed'.format(order)
#     np.savetxt(search_space_out_file, search_space_out, fmt='%s', delimiter='\t')

4
5
6
7
8
9
10
11


In [25]:
# sort and map
"""
sort-bed hilbert_regions_order_4_CT20230718.dhs.cs.sfc.ranks.search_space.ft_2.rt_21.bed > hilbert_regions_order_4_CT20230718.dhs.cs.sfc.ranks.search_space.ft_2.rt_21.sorted.bed
bedmap --echo --echo-map-id hilbert_regions_order_4_CT20230718.dhs.cs.sfc.ranks.search_space.ft_2.rt_21.sorted.bed /home/ctrader/region_similarity/data/rank_based/GREATv4/GREATv4_genes.extended_domain_1000kb.gene.CT20230601.sorted.bed > hilbert_regions_order_4_CT20230718.dhs.cs.sfc.ranks.search_space.ft_2.rt_21.sorted.GREATv4_genes_extended_domain_1000kb_mapped.CT20230925.bed
"""

**Parse mapped file and populate binary Geneset x Region matrix**

In [86]:
datadir = '/home/ctrader/region_similarity/data/rank_based/GREATv4/'
in_f = datadir + 'GREATv4_genes.TSS_bp.gene.strand.basal_5kbup_1kbdown.extended_1000kb.CT20230601.bed'
genes_in_great4_coords = np.loadtxt(in_f, dtype=str, delimiter='\t')

In [87]:
# gene_order = np.unique(genes_in_great4_coords[:, 2])
geneset_order = np.unique(list(geneset2genes.keys()))

In [88]:
geneset_order.shape

(7751,)

In [89]:
datadir = '/home/ctrader/region_similarity/data/rank_based/'
path = datadir + 'c5GOBP_geneset_ordered.GPT.1D_UMAP.IJ20230616.tsv'

f = open(path)
lines = f.readlines()
f.close()

GPT_order_full = []
for line in lines[1:]:
    line = line.strip().split('\t')
    GPT_order_full.append(line)
GPT_order_full = np.array(GPT_order_full)

GPT_order = GPT_order_full[:, 1]

In [90]:
# limiting genesets to just those we have an ordering for
geneset_order = geneset_order[np.isin(geneset_order, GPT_order)]

In [91]:
geneset_order.shape

(7747,)

In [126]:
# for each order, create a search space X geneset binary matrix

datadir = '/home/ctrader/region_similarity/data/rank_based/hilbert/sfc/20230718/'
for order in [4, 5, 6, 7, 8, 9, 10, 11]:
    
    mapped_f = datadir + 'hilbert_regions_order_{}.dhs.cs.sfc.ranks.search_space.ft_2.rt_21.sorted.GREATv4_genes_extended_domain_1000kb_mapped.CT20230925.bed'.format(order)

    f = open(mapped_f)
    lines = f.readlines()
    f.close()

    # binary matrix for genes located in each search space region
    search_space_geneset_bin = np.zeros((len(lines), len(geneset_order)), dtype=int)
    
    print(order, search_space_geneset_bin.shape)

    # for each search space region, populate binary matrix
    for i in range(len(lines)):
        line = lines[i]
        line = re.split('[\t|;]', line.strip())
        mapped_genes = line[4:]

        mapped_gene_genesets = np.array([])
        for g in mapped_genes:
            if g in gene2genesets:
                mapped_gene_genesets = np.append(mapped_gene_genesets, gene2genesets[g])
        mapped_gene_genesets = np.unique(mapped_gene_genesets)

        gs_inds = np.where(np.isin(geneset_order, mapped_gene_genesets))[0]

        search_space_geneset_bin[i, gs_inds] = 1
        
        
    out_f = datadir + 'hilbert_regions_order_{}.dhs.cs.sfc.ranks.search_space.ft_2.rt_21.mapped_extended_domain_GREATv4_gene_c5GOBP_selected_genesets.bin.CT20230927.npz'.format(order)
    np.savez(out_f, search_space_geneset_bin=search_space_geneset_bin.astype(bool), geneset_order=geneset_order)

4 (2384, 7747)
5 (6473, 7747)
6 (10752, 7747)
7 (13437, 7747)
8 (14210, 7747)
9 (39350, 7747)
10 (132467, 7747)
11 (272639, 7747)


**Run a query**

In [11]:
def production_post_request(coords, order, num_sim_regions):
    prod_url = 'https://explore.altius.org:5001/simsearch'
    post_body = {
        'location': coords,
        'includedFactors': '[' + ','.join(np.arange(16).astype(str)) + ']',
        'selectedFactors': '[' + ','.join(np.arange(16).astype(str)) + ']',
        'roiURL': 'False',
        'scale': order,
        'queryFactorThresh': '0.5',
        'numSimilarRegions': str(num_sim_regions),
        'regionMethod': 'hilbert_sfc',
        'fusionWeight': '0.1',
    }
    response = requests.post(prod_url, json=post_body)
    return response

def query_sim_hypergeom(region_inds, N, M, n, geneset_order, search_space_geneset_bin):
    k = np.sum(search_space_geneset_bin[region_inds], axis=0)
    inds_to_test = np.where(k)[0]

    test = hypergeom.sf(k[inds_to_test], M, n[inds_to_test], N)
    tested_genesets = geneset_order[inds_to_test]
    test = test * len(geneset_order)  # bonferroni correction
    test = np.where(test > 1, 1, test)
    return tested_genesets, inds_to_test, test

In [37]:
# query_coords = 'chr10:100141000-100166000' # pretty good, dl=1
# query_coords = 'chr4:173514000-173539000' # good at dl=7
# query_coords = 'chr14:28755875-28780934' # decent at dl=3, dl=5
# query_coords = 'chr2:88112000-88137000' # great at dl=1
query_coords = 'chr4:173522944-173539328'
gene = 'HAND2'
# query_coords = 'chr14:28753920-28770304'
# gene = 'FOXG1'
# query_coords = 'chrX:49250304-49266688'
# gene = 'FOXP3'
# query_coords = 'chr22:41484288-41549824'
# gene = 'ACO2'
# query_coords = 'chr5:173211648-173277184'
# gene='NKX2-5'



order = 7
response = production_post_request(query_coords, order, num_sim_regions=20)
query_full_data = json.loads(response.text)

# # find best detail level
# prod_ranks = []
# for i in range(len(query_full_data)):
#     prod_ranks.append(query_full_data[i][0]['prod_rank'])
# dl = np.argmax(prod_ranks) + 1

query_sim_region_inds = []
query_sim_region_coords = []
for i in range(1, len(query_full_data)):
    coord = re.split('[:-]', query_full_data[i]['coordinates'])
    query_sim_region_coords.append(coord)
    query_sim_region_inds.append(query_full_data[i]['region_ind'])
    
# query_sim_region_coords = re.split('[:-]', query_full_data[i]['coordinates'])
# query_sim_region_inds = query_full_data[i]['region_ind']
    
    
num_sim_regions = len(query_sim_region_coords)
num_regions = search_space_geneset_bin.shape[0]
gs_region_counts = np.sum(search_space_geneset_bin, axis=0)
    
query_tested_gs, query_tested_gs_inds, query_tested_gs_pval = query_sim_hypergeom(
    region_inds=query_sim_region_inds, 
    N=20, # num_sim_regions
    M=num_regions, 
    n=gs_region_counts, 
    geneset_order=geneset_order,
    search_space_geneset_bin=search_space_geneset_bin
)

In [38]:
len(query_sim_region_coords)

20

In [147]:
query_tested_gs[np.argsort(query_tested_gs_pval)], np.sort(query_tested_gs_pval)

(array(['GOBP_EMBRYO_DEVELOPMENT',
        'GOBP_POSITIVE_REGULATION_OF_TRANSCRIPTION_BY_RNA_POLYMERASE_II',
        'GOBP_EMBRYONIC_MORPHOGENESIS', ...,
        'GOBP_MAINTENANCE_OF_PROTEIN_LOCATION_IN_CELL',
        'GOBP_MAST_CELL_ACTIVATION', 'GOBP_WOUND_HEALING'], dtype='<U148'),
 array([7.60549079e-08, 4.55652029e-07, 1.02230509e-06, ...,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00]))

In [22]:
t = time.time()
datadir = '/home/ctrader/region_similarity/data/rank_based/hilbert/sfc/20230718/'
ssbin_path = os.path.join(datadir, 'hilbert_regions_order_7.dhs.cs.sfc.ranks.search_space.ft_2.rt_21.mapped_extended_domain_GREATv4_gene_c5GOBP_selected_genesets.bin.CT20230927.npz')
search_space_geneset_bin = np.load(ssbin_path)['search_space_geneset_bin']
geneset_order = np.load(ssbin_path)['geneset_order']
print(time.time() - t)

1.1715915203094482


In [56]:
datadir = '/home/ctrader/region_similarity/data/rank_based/hilbert/sfc/20230718/'
for i in range(4, 12):
    print(i)
    
    t = time.time()
    ssbin_path = os.path.join(datadir, 'hilbert_regions_order_{}.dhs.cs.sfc.ranks.search_space.ft_2.rt_21.mapped_extended_domain_GREATv4_gene_c5GOBP_selected_genesets.bin.CT20230927.npz'.format(i))
    search_space_geneset_bin = np.load(ssbin_path)['search_space_geneset_bin']
    geneset_order = np.load(ssbin_path)['geneset_order']
    print(time.time() - t)
    
    t = time.time()
    ssbin_path = os.path.join(datadir, 'hilbert_regions_order_{}.dhs.cs.sfc.ranks.search_space.ft_2.rt_21.mapped_extended_domain_GREATv4_gene_c5GOBP_selected_genesets.bin.compressed.CT20231023.npz'.format(i))
    search_space_geneset_bin = np.load(ssbin_path)['search_space_geneset_bin']
    geneset_order = np.load(ssbin_path)['geneset_order']
    print(time.time() - t)
    
    print()

4
0.2449328899383545
0.08727002143859863

5
0.12620925903320312
0.21328353881835938

6
0.1948695182800293
0.33963584899902344

7
0.24524140357971191
0.4563307762145996

8
0.25789356231689453
0.47887182235717773

9
0.6789553165435791
1.2495369911193848

10
2.265763759613037
4.094825744628906

11
4.550128936767578
8.267193794250488



In [55]:
datadir = '/home/ctrader/region_similarity/data/rank_based/hilbert/sfc/20230718/'
for i in range(4, 12):
    print(i)
    
    ssbin_path = os.path.join(datadir, 'hilbert_regions_order_{}.dhs.cs.sfc.ranks.search_space.ft_2.rt_21.mapped_extended_domain_GREATv4_gene_c5GOBP_selected_genesets.bin.CT20230927.npz'.format(i))
    search_space_geneset_bin = np.load(ssbin_path)['search_space_geneset_bin']
    geneset_order = np.load(ssbin_path)['geneset_order']
    
    out_path = os.path.join(datadir, 'hilbert_regions_order_{}.dhs.cs.sfc.ranks.search_space.ft_2.rt_21.mapped_extended_domain_GREATv4_gene_c5GOBP_selected_genesets.bin.compressed.CT20231023.npz'.format(i))
    np.savez_compressed(out_path, search_space_geneset_bin=search_space_geneset_bin, geneset_order=geneset_order)

4
5
6
7
8
9
10
11


In [53]:
search_space_geneset_bin.shape

(2384, 7747)

In [ ]:
genesets_reordered

In [ ]:
terms_to_remove = np.array(['GOBP'])
genesets_reordered = GPT_order
window_size=100
label_padding=120
label_scale=3.5

spectrum_values = np.zeros(len(genesets_reordered))
spectrum_gs = np.zeros(len(genesets_reordered), dtype='<U' + str(max_len))
spectrum_value_inds = []
for gs, pval in zip(query_sig_gs, query_sig_gs_pval):
    ind = np.where(genesets_reordered == gs)[0][0]
    spectrum_values[ind] = -np.log10(pval)
    spectrum_gs[ind] = gs
    spectrum_value_inds.append(ind)
spectrum_value_inds = np.sort(spectrum_value_inds)

# find the max over window at each position
spectrum_max = np.zeros(len(genesets_reordered))
for i in range(len(genesets_reordered)):
    lower_ind = np.max(
            [0,
            i - int(window_size / 2)]
        )
    upper_ind = np.min(
            [i + int(window_size / 2), 
            len(genesets_reordered)]
        )
    test_inds = np.arange(
        lower_ind, 
        upper_ind
    )
    window_values = spectrum_values[test_inds]
    window_max = np.max(window_values)
    spectrum_max[i] = window_max
    
# average the max values over window each position to smooth
spectrum_mean = np.zeros(len(genesets_reordered))
for i in range(len(genesets_reordered)):
    lower_ind = np.max(
            [0,
            i - int(window_size / 2)]
        )
    upper_ind = np.min(
            [i + int(window_size / 2), 
            len(genesets_reordered)]
        )
    test_inds = np.arange(
        lower_ind, 
        upper_ind
    )
    window_values = spectrum_max[test_inds]
    window_mean = np.mean(window_values)
    spectrum_mean[i] = window_mean
    

    
assignment_order = np.argsort(-spectrum_values)
assignment_order = assignment_order[np.isin(assignment_order, spectrum_value_inds)]

spectrum_gs_labels = np.zeros(len(genesets_reordered), dtype='<U' + str(max_len))
for a_ind in assignment_order:
    label = spectrum_gs[a_ind]
    if (
        (label != '') & 
        (np.count_nonzero(spectrum_gs_labels[
            np.max([0, (a_ind - label_padding)]):
            np.min([(a_ind + label_padding), len(genesets_reordered)])
        ]) == 0)
    ):
        label_l = np.array(label.split('_'))
        label_l = label_l[~np.isin(label_l, terms_to_remove)]
        label = ' '.join(label_l)
        spectrum_gs_labels[a_ind] = label
inds_to_label = np.where(spectrum_gs_labels)[0]

fig, axes = plt.subplots(2, 1, figsize=(12, 2), gridspec_kw={'height_ratios': [4, 1]})  
fig.patch.set_facecolor('white')
plt.subplots_adjust(hspace=0)
axes.ravel()

axes[0].fill_between(range(len(spectrum_mean)), spectrum_mean, color='k')
axes[0].set_xticks([])
axes[0].set_xlim([0, len(spectrum_mean)])
axes[0].set_ylim([0, np.max(spectrum_mean)])
axes[0].text(600, np.max(spectrum_mean) * 1.1, 'Development')
axes[0].text(2100, np.max(spectrum_mean) * 1.1, 'Immune')
axes[0].text(3400, np.max(spectrum_mean) * 1.1, 'Signaling')
axes[0].text(4300, np.max(spectrum_mean) * 1.1, 'Ion Transport')
axes[0].text(5200, np.max(spectrum_mean) * 1.1, 'Housekeeping')
axes[0].text(6400, np.max(spectrum_mean) * 1.1, 'Lipid')
axes[0].text(7000, np.max(spectrum_mean) * 1.1, 'Metabolism')




colors = cm.get_cmap('nipy_spectral', len(genesets_reordered))
matplotlib.colorbar.ColorbarBase(axes[1], cmap=colors, orientation='horizontal').set_ticks(inds_to_label / len(spectrum_mean))
axes[1].set_xticklabels(spectrum_gs_labels[inds_to_label], rotation=45, ha='right')

for xlabel, size in zip(axes[1].xaxis.get_ticklabels(), spectrum_values[inds_to_label]):
    xlabel.set_fontsize(np.min([size*label_scale, 20]))

fig.suptitle('GPT Order: %s (%s), Detail Level %s, Order %s' %(gene, query_coords, dl, order), y=1.15)